##In this notebook, we calibrate the HGR GR4J model. Note that the technique is the same ofr the GRHyMoLAP model.

# IMPORT LIBRARIES

In [ ]:
!pip install git+https://github.com/kratzert/RRMPG.git

  Cloning https://github.com/kratzert/RRMPG.git to /tmp/pip-req-build-t99ze0ae
  Running command git clone --filter=blob:none --quiet https://github.com/kratzert/RRMPG.git /tmp/pip-req-build-t99ze0ae
  Resolved https://github.com/kratzert/RRMPG.git to commit 7de78c25acc1c255d2acaf739d65e9ce7bbd60c3
  Preparing metadata (setup.py) ... done


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pandas import read_csv
import math
import random
from datetime import datetime, timedelta
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

import zipfile
import os
from scipy.optimize import minimize
from google.colab import files

from rrmpg.models import GR4J

In [ ]:
from google.colab import drive

# Monter Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##CAMELS-DATA from my Python library.

In [ ]:
pip install aqua-fetch

In [ ]:
from aqua_fetch import RainfallRunoff

#
rr = RainfallRunoff("CAMELS_FR")

/usr/local/lib/python3.12/dist-packages/aqua_fetch/rr/utils.py:127: UserWarning: netCDF4 module is not installed. Please install it to save data in netcdf format
  warnings.warn(msg, UserWarning)



        Not downloading the data since the directory 
        /usr/local/lib/python3.12/dist-packages/aqua_fetch/data/CAMELS/CAMELS_FR already exists.
        Use overwrite=True to remove previously saved files and download again
P392252001  not found in site_general_attributes.csv
P613402001  not found in site_general_attributes.csv


In [ ]:
meta, ds = rr.fetch()

Read 654 stations for 22 dyn features in 62.91 seconds with 2 cpus.


In [ ]:
import xarray as xr

print(ds)              # aperçu global
#print(ds.coords)       # coordonnées (temps, variables, stations)
#print(ds['912101A'].head())  # exemple : station 5

<xarray.Dataset> Size: 2GB
Dimensions:           (time: 18993, dynamic_features: 22)
Coordinates:
  * time              (time) datetime64[ns] 152kB 1970-01-01 ... 2021-12-31
  * dynamic_features  (dynamic_features) object 176B 'q_cms_obs' ... 'airtemp...
Data variables: (12/654)
    A105003001        (time, dynamic_features) float64 3MB 1.65e+03 ... 13.4
    A107020001        (time, dynamic_features) float64 3MB nan nan ... 4.1 13.3
    A112020001        (time, dynamic_features) float64 3MB 1.04e+03 ... 13.4
    A116003002        (time, dynamic_features) float64 3MB nan nan ... 4.3 13.3
    A140202001        (time, dynamic_features) float64 3MB nan nan ... 6.8 12.4
    A202030001        (time, dynamic_features) float64 3MB nan nan ... 7.1 13.2
    ...                ...
    Y661401001        (time, dynamic_features) float64 3MB 860.0 0.441 ... 14.3
    Y781000101        (time, dynamic_features) float64 3MB nan nan ... 6.4 15.8
    Y862000101        (time, dynamic_features) float64 3MB 

In [ ]:
print(ds.dynamic_features.values)

['q_cms_obs' 'q_mm_obs' 'tsd_val_s' 'tsd_val_q' 'tsd_val_m' 'tsd_val_c'
 'tsd_val_i' 'pcp_mm' 'pcp_mm_solfrac' 'airtemp_C_mean' 'pet_mm_ou'
 'pet_mm_pe' 'pet_mm_pm' 'windspeed_mps' 'spechum_gkg' 'lwdownrad_wm2'
 'solrad_wm2' 'tsd_swi_gr' 'tsd_swi_isba' 'tsd_swe_isba' 'airtemp_C_min'
 'airtemp_C_max']


In [ ]:
# Full period
print(ds["time"].values[0], ds["time"].values[-1])

1970-01-01T00:00:00.000000000 2021-12-31T00:00:00.000000000


#Period used

In [ ]:
ds_recent = ds.sel(time=slice("2000-01-01", "2021-12-31"))

# List of all stations
all_stations = list(ds_recent.keys())

# Selection of features of interest
features = ["q_mm_obs", "pcp_mm", "pet_mm_pm"]

# Count stations without NaN
valid_stations = []

for station_id in all_stations:
    subset = ds_recent[station_id].sel(dynamic_features=features)
    df = subset.to_dataframe().reset_index().pivot(
        index="time", columns="dynamic_features", values=station_id
    )

    if df[features].isna().sum().max() == 0:  # no missing values
        valid_stations.append(station_id)

GRHyMoLAP

In [ ]:
from numba import njit
from scipy.optimize import minimize
import numpy as np

# ============================================
# GAUSSIAN NEGATIVE LOG-LIKELIHOOD
# sigma = a + b * Qsim
# ============================================

@njit
def Gaussian_NLL(obs, sim, a, b):

    loss = 0.0
    count = 0

    for i in range(len(obs)):

        sigma = a + b * sim[i]

        if (
            not np.isnan(obs[i])
            and not np.isnan(sim[i])
            and sigma > 1e-8
        ):

            diff = obs[i] - sim[i]

            loss += (diff * diff) / (2.0 * sigma * sigma)
            loss += np.log(sigma)

            count += 1

    if count == 0:
        return np.nan

    return loss / count


# ============================================
# OBJECTIVE FUNCTION
# ============================================

def objective(params, Q_obs_train, P_train, ET_train):

    x1, x2, x3, x4, a, b = params

    model = GR4J()

    model.set_params({
        "x1": x1,
        "x2": x2,
        "x3": x3,
        "x4": x4,
    })

    try:

        Q_sim = model.simulate(
            P_train,
            ET_train
        ).flatten()

        loss = Gaussian_NLL(
            Q_obs_train,
            Q_sim,
            a,
            b
        )

        if np.isfinite(loss):
            return loss

    except Exception:
        pass

    return 1e9


# ============================================
# SETTINGS
# ============================================

b1_ratio = 0.6
max_missing_ratio = 0.05

stations = ['K066331001', 'A231020001']

results_GR4J_cv = {}

# ============================================
# MAIN LOOP
# ============================================

i = 0

for station_id in stations:

    i += 1

    print(f"\n=== Station {station_id} ===, Number = {i}")

    Q_obs = ds_recent[station_id].sel(
        dynamic_features="q_mm_obs"
    ).to_numpy()

    P = ds_recent[station_id].sel(
        dynamic_features="pcp_mm"
    ).to_numpy()

    ET = ds_recent[station_id].sel(
        dynamic_features="pet_mm_pm"
    ).to_numpy()

    N = len(Q_obs)

    if N == 0 or np.all(np.isnan(Q_obs)):
        continue

    missing_ratio = np.sum(np.isnan(Q_obs)) / N

    if missing_ratio > max_missing_ratio:
        continue

    b1 = int(N * b1_ratio)

    # ============================================
    # OPTIMIZATION
    # ============================================

    initial_guesses = [

        [300, 0, 100, 1.5, 0.01, 0.10],
        [800, 2, 300, 2.5, 0.05, 0.20],
        [1500, -2, 500, 1.0, 0.10, 0.15],
        [2500, 5, 800, 3.5, 0.02, 0.05],
        [500, -5, 200, 4.0, 0.05, 0.10],

    ]

    bounds = [

        (1, 3500),      # x1
        (-15, 15),      # x2
        (1, 1200),      # x3
        (0.5, 5),       # x4
        (1e-8, None),   # a
        (1e-8, None),   # b

    ]

    best_res = None
    best_val = np.inf

    for guess in initial_guesses:

        res = minimize(

            objective,

            guess,

            args=(
                Q_obs[:b1],
                P[:b1],
                ET[:b1],
            ),

            method="Nelder-Mead",

            bounds=bounds,

            options={
                "maxiter": 2500,
                "disp": False,
            },

        )

        if res.fun < best_val:

            best_val = res.fun
            best_res = res

    if best_res is None:
        continue

    x1, x2, x3, x4, a, b = best_res.x

    # ============================================
    # FULL SIMULATION
    # ============================================

    model = GR4J()

    model.set_params({

        "x1": x1,
        "x2": x2,
        "x3": x3,
        "x4": x4,

    })

    Qsim_cv = model.simulate(
        P,
        ET
    ).flatten()

    print(f"Estimated a = {a:.5f}")
    print(f"Estimated b = {b:.5f}")

    # ============================================
    # STORE
    # ============================================

    results_GR4J_cv[station_id] = {

        "params": [x1, x2, x3, x4, a, b],

        "Qsim": Qsim_cv,
        "Q_obs": Q_obs,

        "a_cv": a,
        "b_cv": b

    }

print(
    f"\n✅ Simulation completed for {len(results_GR4J_cv)} valid basins."
)


=== Station A105003001 ===, Number = 1
Estimated a = 0.25399
Estimated b = 0.26662

=== Station A107020001 ===, Number = 2
Estimated a = 0.25701
Estimated b = 0.24843

=== Station A112020001 ===, Number = 3

=== Station A116003002 ===, Number = 4
Estimated a = 0.26289
Estimated b = 0.24383

=== Station A140202001 ===, Number = 5
Estimated a = 0.98128
Estimated b = 0.39590

=== Station A202030001 ===, Number = 6
Estimated a = 0.75368
Estimated b = 0.16213

=== Station A204010101 ===, Number = 7

=== Station A211030001 ===, Number = 8
Estimated a = 0.70509
Estimated b = 0.20092

=== Station A212020002 ===, Number = 9

=== Station A231020001 ===, Number = 10
Estimated a = 0.31037
Estimated b = 0.29003

=== Station A234021001 ===, Number = 11
Estimated a = 0.40283
Estimated b = 0.24865

=== Station A251020001 ===, Number = 12
Estimated a = 0.51333
Estimated b = 0.22372

=== Station A270011001 ===, Number = 13

=== Station A273011002 ===, Number = 14
Estimated a = 0.62535
Estimated b = 0.2

In [ ]:
from numba import njit

# ============================================================
# PREDICTION INTERVAL COVERAGE PROBABILITY (PICP)
# ============================================================

@njit
def PICP(Q, L, U):

    inside = 0
    count = 0

    for i in range(len(Q)):

        if (not np.isnan(Q[i]) and
            not np.isnan(L[i]) and
            not np.isnan(U[i])):

            if L[i] <= Q[i] <= U[i]:
                inside += 1

            count += 1

    if count == 0:
        return np.nan

    return inside / count


# ============================================================
# MEAN PREDICTION INTERVAL WIDTH (MPIW)
# ============================================================

@njit
def MPIW(Q, L, U):

    width = 0.0
    count = 0

    for i in range(len(Q)):

        if (not np.isnan(Q[i]) and
            not np.isnan(L[i]) and
            not np.isnan(U[i])):

            width += (U[i] - L[i])

            count += 1

    if count == 0:
        return np.nan

    return width / count

# ============================================================
# INTERVAL SCORE
# ============================================================

@njit
def IS(Q, L, U, alpha=0.1):

    loss = 0.0
    count = 0

    for i in range(len(Q)):

        if (not np.isnan(Q[i]) and
            not np.isnan(L[i]) and
            not np.isnan(U[i])):

            width = U[i]-L[i]

            under = max(L[i]-Q[i],0.0)
            over  = max(Q[i]-U[i],0.0)

            loss += width + (2.0/alpha)*(under+over)

            count += 1

    if count == 0:
        return np.nan

    return loss/count

In [ ]:
import numpy as np

# ============================================================
# CONSTANT
# ============================================================

z = 1.645   # 90% prediction interval

# ============================================================
# GLOBAL METRICS
# ============================================================

all_picp = []
all_mpiw = []
all_is = []

# ============================================================
# LOOP OVER BASINS
# ============================================================

for station_id in results_GR4J_cv.keys():

    res = results_GR4J_cv[station_id]

    Qsim = res["Qsim"]
    Qobs = res["Q_obs"]

    a = res["a_cv"]
    b = res["b_cv"]

    # =========================================================
    # UNCERTAINTY BOUNDS
    # =========================================================

    sigma = a + b * Qsim

    Q5 = np.maximum(0.0, Qsim - z * sigma)
    Q95 = Qsim + z * sigma

    # =========================================================
    # METRICS
    # =========================================================

    is_score = IS(Qobs, Q5, Q95)
    picp = PICP(Qobs, Q5, Q95)
    mpiw = MPIW(Qobs, Q5, Q95)

    # Store
    results_GR4J_cv[station_id].update({
        "Q5": Q5,
        "Q95": Q95,
        "IS": is_score,
        "PICP": picp,
        "MPIW": mpiw
    })

    all_is.append(is_score)
    all_picp.append(picp)
    all_mpiw.append(mpiw)

# ============================================================
# SUMMARY
# ============================================================

all_is = np.array(all_is)
all_picp = np.array(all_picp)
all_mpiw = np.array(all_mpiw)

print("\n================ Overall uncertainty metrics ================\n")

for name, values in [
    ("PICP", all_picp),
    ("MPIW", all_mpiw),
    ("IS", all_is),
]:

    print(f"{name}")
    print(f"  Mean   : {np.mean(values):.4f}")
    print(f"  Median : {np.median(values):.4f}")
    print(f"  Min    : {np.min(values):.4f}")
    print(f"  Max    : {np.max(values):.4f}")
    print()


================ Overall uncertainty metrics ================

PICP
  Mean   : 0.9438
  Median : 0.9465
  Min    : 0.8706
  Max    : 0.9775

MPIW
  Mean   : 1.9070
  Median : 1.4867
  Min    : 0.4237
  Max    : 7.8504

IS
  Mean   : 2.7544
  Median : 2.1492
  Min    : 0.6365
  Max    : 11.9621



In [ ]:
from google.colab import files

# ============================================================
# KEEP ONLY VALID BASINS
# ============================================================

stations = [
    k for k, v in results_GR4J_cv.items()
    if (
        not np.isnan(v["IS"])
        and not np.isnan(v["PICP"])
        and not np.isnan(v["MPIW"])
    )
]

# ============================================================
# CREATE DATAFRAME
# ============================================================

metrics_df = pd.DataFrame({
    "Station": stations,
    "IS": [
        v["IS"] for v in results_GR4J_cv.values()
        if (
            not np.isnan(v["IS"])
            and not np.isnan(v["PICP"])
            and not np.isnan(v["MPIW"])
        )
    ],
    "PICP": [
        v["PICP"] for v in results_GR4J_cv.values()
        if (
            not np.isnan(v["IS"])
            and not np.isnan(v["PICP"])
            and not np.isnan(v["MPIW"])
        )
    ],
    "MPIW": [
        v["MPIW"] for v in results_GR4J_cv.values()
        if (
            not np.isnan(v["IS"])
            and not np.isnan(v["PICP"])
            and not np.isnan(v["MPIW"])
        )
    ]
})

# ============================================================
# SAVE
# ============================================================

metrics_df.to_csv("GR4J_CV_metrics.csv", index=False)

# ============================================================
# DOWNLOAD
# ============================================================

#files.download("GR4J_CV_metrics.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>